# Electronic Structure for Atomistic Simulation

This notebook introduces the key concepts behind electronic structure calculations using Density Functional Theory (DFT). Understanding these concepts is essential for interpreting the results of your calculations and knowing the limitations of the method.

**Topics covered:**
- Periodic boundary conditions (PBC)
- The exchange-correlation functional
- K-point sampling
- Spin polarisation and magnetisation
- Fundamental limits of DFT

---


## Lecture Slides

The slides for this tutorial are embedded below.
[📥 Download slides (.pptx)](https://github.com/NU-CEM/Atomistic_Simulation/raw/2026/slides/Tutorial7_ElectronicStructure.pptx) &nbsp;|&nbsp; [Open in full screen](SHAREPOINT_EMBED_URL_HERE)

<iframe
  src="SHAREPOINT_EMBED_URL_HERE"
  width="100%"
  height="480"
  frameborder="0"
  allowfullscreen="true">
</iframe>

> **How to embed:** Upload the .pptx to PowerPoint Online (SharePoint/OneDrive) → File → Share → Embed → copy the `src="..."` URL and replace `SHAREPOINT_EMBED_URL_HERE` above.
> The download link already points to `slides/Tutorial7_ElectronicStructure.pptx` on GitHub — just commit the file to that path.

---

## 1. Periodic Boundary Conditions (PBC)

Real crystals contain ~102³ atoms — far too many to simulate directly. Instead, we simulate a small **unit cell** and apply **periodic boundary conditions (PBC)**: the simulation cell is tiled infinitely in all directions, so every atom "sees" the same environment.

This is exact for a perfect crystal, but introduces an **approximation for defects** — the defect interacts with its periodic images in neighbouring cells.

```{figure} ../images/pbc.png
:width: 60%
Periodic boundary conditions: the simulation cell (bold) is surrounded by identical copies.
```

In ASE, PBC is controlled per-axis:


In [ ]:
from ase.build import bulk
from ase import Atoms
import numpy as np

# 3D periodic (bulk crystal)
diamond = bulk('C', 'diamond', a=3.57)
print(f"Diamond PBC: {diamond.pbc}")

# 2D periodic (monolayer with vacuum)
a = 2.504
hbn = Atoms('BN',
            scaled_positions=[[0, 0, 0.5], [2/3, 1/3, 0.5]],
            cell=[[a, 0, 0], [-a/2, a*np.sqrt(3)/2, 0], [0, 0, 20.0]],
            pbc=[True, True, False])
print(f"hBN monolayer PBC: {hbn.pbc}")
print(f"hBN cell (Å): {hbn.cell.lengths()}")


### The supercell approximation for defects

When we introduce a defect, we use a **supercell** — a larger cell containing many unit cells — so that the defect is surrounded by enough bulk material and its interaction with periodic images is small.

The key quantity is the **minimum image distance**: the distance between the defect and its nearest periodic copy.


In [ ]:
from ase.build import make_supercell

# Compare supercell sizes for diamond
prim = bulk('C', 'diamond', a=3.57)

for n in [2, 3, 4]:
    sc = make_supercell(prim, np.diag([n, n, n]))
    min_image_dist = sc.cell[0, 0]   # cubic cell: min image = cell edge
    print(f"{n}x{n}x{n}: {len(sc):3d} atoms, "
          f"min image distance = {min_image_dist:.2f} Å")

print("\nRule of thumb: min image distance > 10 Å for isolated defect")


---
## 2. The Exchange-Correlation Functional

The foundation of DFT is the **Hohenberg-Kohn theorem**: the ground state energy of a many-electron system is uniquely determined by its electron density $n(\mathbf{r})$.

The total energy is:

$$E[n] = T_s[n] + E_{\text{ext}}[n] + E_{\text{Hartree}}[n] + E_{\text{XC}}[n]$$

where:
- $T_s$ = kinetic energy of non-interacting electrons
- $E_{\text{ext}}$ = electron-nuclear interaction
- $E_{\text{Hartree}}$ = classical electron-electron repulsion
- $E_{\text{XC}}$ = **exchange-correlation energy** — contains everything else

The problem: **$E_{\text{XC}}$ is unknown exactly**. We must approximate it.

### The XC functional ladder (Jacob's ladder)

| Rung | Functional | Example | Cost | Accuracy |
|------|-----------|---------|------|----------|
| 1 | LDA | PW92 | Low | Poor gaps |
| 2 | GGA | **PBE** | Low | Standard |
| 3 | meta-GGA | SCAN | Medium | Better gaps |
| 4 | Hybrid | **HSE06** | High | Good gaps |
| 5 | RPA | | Very high | Excellent |

**PBE** is the workhorse of atomistic simulation — fast and reliable for geometries and energetics, but **systematically underestimates bandgaps** by 30-50%.

**HSE06** mixes 25% exact (Hartree-Fock) exchange with 75% PBE — much better bandgaps but 10-100× more expensive.


In [ ]:
# Demonstrate PBE bandgap underestimation
# (values from literature — we can't run HSE06 here)

materials = {
    'Diamond':  {'PBE': 4.2,  'HSE06': 5.3,  'Exp': 5.47},
    'GaN':      {'PBE': 1.7,  'HSE06': 3.1,  'Exp': 3.40},
    'AlN':      {'PBE': 4.0,  'HSE06': 5.7,  'Exp': 6.00},
    'hBN mono': {'PBE': 4.7,  'HSE06': 6.2,  'Exp': 6.10},
    '4H-SiC':   {'PBE': 2.2,  'HSE06': 3.0,  'Exp': 3.23},
}

import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(materials))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
labels = list(materials.keys())
pbe   = [materials[m]['PBE']   for m in labels]
hse   = [materials[m]['HSE06'] for m in labels]
exp   = [materials[m]['Exp']   for m in labels]

ax.bar(x - width, pbe, width, label='PBE',   color='steelblue', alpha=0.8)
ax.bar(x,         hse, width, label='HSE06', color='tomato',    alpha=0.8)
ax.bar(x + width, exp, width, label='Exp.',  color='green',     alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Band gap (eV)', fontsize=12)
ax.set_title('Bandgap underestimation: PBE vs HSE06 vs experiment', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("PBE underestimates gaps by 30-50% for wide-gap materials.")
print("This matters for defect level positions — if the gap is wrong,")
print("the defect level position relative to band edges is also wrong.")


---
## 3. K-point Sampling

In a periodic solid, Bloch's theorem tells us that electronic wavefunctions have the form:

$$\psi_{n\mathbf{k}}(\mathbf{r}) = e^{i\mathbf{k}\cdot\mathbf{r}} u_{n\mathbf{k}}(\mathbf{r})$$

where $\mathbf{k}$ is a vector in the **Brillouin zone (BZ)**. To compute the total energy we must integrate over all $\mathbf{k}$ in the BZ — in practice we sample on a discrete mesh.

### Convergence testing

**Always test k-point convergence** — using too few k-points gives wrong energies and gaps.


In [ ]:
from ase.build import bulk
from gpaw import GPAW, Mixer

# Demonstrate k-point convergence (schematic — uses pre-computed values)
# In a real calculation you would run GPAW for each k-mesh

# Pre-computed PBE total energies for diamond (eV/atom) at different k-meshes
# (representative values from literature)
k_meshes = ['2x2x2', '4x4x4', '6x6x6', '8x8x8', '10x10x10']
energies  = [-157.12, -157.43, -157.51, -157.53, -157.54]   # eV/atom (schematic)
gaps      = [3.8,      4.1,     4.2,     4.2,     4.2]       # eV (schematic)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(len(k_meshes)), energies, 'o-', color='steelblue', lw=2)
axes[0].set_xticks(range(len(k_meshes)))
axes[0].set_xticklabels(k_meshes, fontsize=10)
axes[0].set_ylabel('Total energy (eV/atom)', fontsize=12)
axes[0].set_title('K-point convergence: total energy', fontsize=11)
axes[0].axhline(energies[-1], color='red', ls='--', lw=1, label='Converged')
axes[0].legend()

axes[1].plot(range(len(k_meshes)), gaps, 's-', color='tomato', lw=2)
axes[1].set_xticks(range(len(k_meshes)))
axes[1].set_xticklabels(k_meshes, fontsize=10)
axes[1].set_ylabel('Band gap (eV)', fontsize=12)
axes[1].set_title('K-point convergence: band gap', fontsize=11)
axes[1].axhline(gaps[-1], color='red', ls='--', lw=1, label='Converged')
axes[1].legend()

plt.suptitle('Diamond — k-point convergence (schematic)', fontsize=12)
plt.tight_layout()
plt.show()

print("Rule of thumb: converge energy to 1 meV/atom.")
print("For supercells (defect calculations), Gamma-point only is often sufficient")
print("because the BZ is already small due to zone folding.")


### K-points for supercells

For a **supercell** with N×N×N primitive cells, the BZ is N times smaller in each direction. A Γ-only (1×1×1) sampling of a 3×3×3 supercell is roughly equivalent to a 3×3×3 sampling of the primitive cell — adequate for many defect calculations.

This is why we can use Γ-only for defect supercells and still get reasonable results.

---
## 4. Spin Polarisation and Magnetisation

By default, DFT assumes spin-up and spin-down electrons have the same spatial distribution (spin-paired). For systems with unpaired electrons — magnetic materials, open-shell defects — we must use **spin-polarised DFT**, where the two spin channels are treated independently.

The **magnetic moment** is:

$$m = \int [n_{\uparrow}(\mathbf{r}) - n_{\downarrow}(\mathbf{r})] \, d\mathbf{r}$$

For a spin-1 system (like NV(-)), m = 2. For spin-1/2, m = 1.


## Running GPAW in Parallel

GPAW supports MPI parallelisation with no changes to your script. Simply run with `mpirun`:

```bash
# Single core (default)
python defect_dos.py

# 4 cores — recommended for defect supercells
mpirun -n 4 gpaw python defect_dos.py

# Check how many cores your machine has
python -c "import os; print(os.cpu_count())"
```

GPAW automatically distributes the real-space grid, k-points, and bands across cores. For a ~60 atom supercell with Gamma-only k-points, expect roughly **2-3x speedup** with 4 cores (parallelisation is over the real-space grid, not k-points, so scaling is imperfect).

For the pristine band structure calculation with a 6x6x6 k-mesh, parallelisation over k-points is more efficient and you may see closer to **4x speedup** with 4 cores.

> **Note:** Use `gpaw python` not plain `python` when running with `mpirun` — this ensures GPAW's MPI environment is correctly initialised.


In [ ]:
import os

n_cores = os.cpu_count()
print(f"This machine has {n_cores} CPU cores available.")
print()
print("Recommended mpirun settings:")
print(f"  mpirun -n {min(4, n_cores)} gpaw python defect_dos.py")
print()
print("Expected speedups (approximate, 60-atom supercell, Gamma-only):")
for n in [1, 2, 4, 8]:
    if n <= n_cores:
        speedup = 1 + (n-1) * 0.6   # rough estimate accounting for overhead
        print(f"  {n} cores: ~{speedup:.1f}x speedup")


In [ ]:
# Spin-polarised calculation example using EMT (proxy for DFT)
# In a real calculation use GPAW with spinpol=True

from ase.build import bulk, make_supercell
from ase import Atoms
import numpy as np

# Build NV centre (neutral for this demo)
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([2, 2, 2]))

from ase.geometry import get_distances
_, D = get_distances(sc.positions, sc.positions, cell=sc.cell, pbc=True)
np.fill_diagonal(D, np.inf)
vac_idx = int(np.argmin(D[0]))
sym = sc.get_chemical_symbols()
sym[0] = 'N'
sc.set_chemical_symbols(sym)
del sc[vac_idx]

# Set initial magnetic moments
# NV(-) has S=1, total moment = 2
magmoms = [0.0] * len(sc)
magmoms[0] = 2.0   # on N atom — will redistribute during SCF
sc.set_initial_magnetic_moments(magmoms)

print(f"System: {sc.get_chemical_formula()}")
print(f"Total initial magnetic moment: {sum(sc.get_initial_magnetic_moments()):.1f} uB")
print(f"\nIn GPAW, use:")
print(f"  GPAW(spinpol=True, charge=-1, ...)")
print(f"  # charge=-1 for NV(-)")
print(f"  # spinpol=True enables separate spin channels")


### Why initial magnetic moments matter

GPAW (like most DFT codes) can get stuck in a **local minimum** with the wrong spin state if started from a bad initial guess. For open-shell defects:

- Always set `spinpol=True`
- Set initial magnetic moments consistent with the expected spin state
- Use a small mixer `beta` (0.02–0.05) to avoid oscillation
- Check the converged magnetic moment in the output

```
iter:  50  ...  +2.001   <-- converged to correct S=1 (moment=2)
iter:  50  ...  +0.001   <-- converged to wrong S=0 — restart with better initial moments
```

---
## 5. Fundamental Limits of DFT

DFT is a **ground state theory**. This has important consequences for quantum optics:

### What DFT can compute:
- Ground state geometry [yes]
- Ground state total energy [yes]
- Electron density [yes]
- Band structure (ground state) [yes]
- Defect formation energies [yes]

### What DFT cannot compute directly:
- **Excited state energies** [no] — need TDDFT, BSE, or Δ-SCF
- **Optical spectra** [no] — need many-body perturbation theory (GW+BSE)
- **Quasiparticle gaps** [no] — PBE gap ≠ optical gap
- **Exciton binding energies** [no]

### The gap problem

The DFT bandgap is a **Kohn-Sham gap** — the difference between the highest occupied and lowest unoccupied Kohn-Sham eigenvalues. This is **not** the same as the **quasiparticle gap** (what you measure in experiment), because:

$$E_{\text{gap}}^{\text{exp}} = E_{\text{gap}}^{\text{KS}} + \Delta_{\text{xc}}$$

where $\Delta_{\text{xc}}$ is the derivative discontinuity of the XC functional, which is zero for LDA/GGA but non-zero in reality.

For quantum optics, this means:
- Defect level positions relative to band edges are approximate
- Zero-phonon line energies require excited state calculations (Δ-SCF)
- Use PBE results as a starting point, not a final answer


In [ ]:
# Summary: what we can and cannot trust from PBE

print("=" * 55)
print("PBE DFT: What to trust for quantum emitter calculations")
print("=" * 55)
print()
print("RELIABLE:")
print("  [yes] Relaxed geometry (bond lengths within ~1%)")
print("  [yes] Relative energetics (formation energy trends)")
print("  [yes] Whether a defect level exists in the gap")
print("  [yes] Qualitative spin state")
print("  [yes] Strain response of defect levels")
print()
print("USE WITH CAUTION:")
print("  ~ Absolute defect level position (±0.5 eV)")
print("  ~ Bandgap (typically 30-50% underestimate)")
print("  ~ Magnetic moment (usually correct qualitatively)")
print()
print("DO NOT TRUST:")
print("  [no] Zero-phonon line energy (need Δ-SCF or TDDFT)")
print("  [no] Optical absorption spectrum")
print("  [no] Excited state geometry")
print()
print("For publication-quality results: use HSE06 or G0W0")
print("For teaching/screening: PBE is a good starting point")


---
## Band Structure and Density of States

## Why Electronic Structure Matters for Quantum Optics

The electronic structure of a material determines:
- **Bandgap** — whether the material can emit visible/NIR light
- **Direct/indirect gap** — whether light emission is allowed by momentum conservation
- **Defect levels** — the energy of in-gap states associated with quantum emitters
- **Effective masses** — carrier transport relevant for device engineering

### Key materials and their bandgaps

| Material | Bandgap (eV) | Type | Application |
|----------|-------------|------|-------------|
| Diamond | 5.47 | Indirect | NV centre host |
| hBN | ~6.0 | Indirect (bulk) / ~6.1 (mono.) | V_B emitter |
| GaN | 3.4 | Direct | LEDs, quantum emitter host |
| AlN | 6.0 | Direct | Deep-UV emitters |
| Si | 1.1 | Indirect | Photonic circuits |
| GaAs | 1.4 | Direct | Quantum dots |


## Band Structure Workflow

Computing a band structure requires two DFT calculations:

1. **Self-consistent field (SCF)** — solve for the electron density on a uniform k-mesh
2. **Non-self-consistent (NSCF)** — compute eigenvalues at k-points along high-symmetry paths (using the fixed density from step 1)

ASE provides `BandPath` objects that enumerate high-symmetry paths for any Bravais lattice:


In [ ]:
from ase.build import bulk
from ase.dft.kpoints import get_special_points, bandpath
import numpy as np
import matplotlib.pyplot as plt

# Get the high-symmetry points for the FCC lattice (used by diamond and GaAs)
diamond_C = bulk('C', 'diamond', a=3.57)
path = diamond_C.cell.bandpath('GXWKGLUWLK', npoints=100)

print("High-symmetry path for diamond/zinc-blende FCC:")
print(f"  Path: {path.path}")
print(f"  Number of k-points: {len(path.kpts)}")
print(f"\nSpecial points:")
sp = path.special_points
for label, kpt in sp.items():
    print(f"  {label}: {kpt}")


In [ ]:
# Hexagonal (wurtzite GaN) Brillouin zone path
gan_wz = bulk('GaN', 'wurtzite', a=3.19, c=5.19)
path_hex = gan_wz.cell.bandpath('GMKGALHA', npoints=100)
print("High-symmetry path for wurtzite (hexagonal):")
print(f"  Path: {path_hex.path}")
print(f"  Special points: {list(path_hex.special_points.keys())}")


## Using GPAW for a Band Structure

Below is the template for computing a GaN band structure with GPAW. This requires GPAW to be installed. We show the code pattern, then display a pre-computed result.

```python
from gpaw import GPAW, PW, FermiDirac
from ase.build import bulk

gan = bulk('GaN', 'wurtzite', a=3.19, c=5.19)

# Step 1: SCF on uniform k-mesh
calc = GPAW(mode=PW(600),
            xc='PBE',
            kpts={'size': (8, 8, 6), 'gamma': True},
            occupations=FermiDirac(0.01),
            txt='gan_scf.txt')
gan.calc = calc
gan.get_potential_energy()
calc.write('gan_scf.gpw')

# Step 2: Band structure along Γ→M→K→Γ→A
path = gan.cell.bandpath('GMKGALHA', npoints=100)
calc_bs = GPAW('gan_scf.gpw').fixed_density(
    kpts=path,
    symmetry='off',
    txt='gan_bs.txt')
gan.calc = calc_bs
gan.get_potential_energy()

bs = calc_bs.band_structure()
bs.plot(emin=-6, emax=8, filename='gan_bandstructure.png', show=True)
```


In [ ]:
# Since GPAW is not available here, we demonstrate with a schematic
# band structure for pedagogical purposes.
# In your own work, replace this with your GPAW/QE output.

def gan_schematic_bands():
    # Schematic GaN-like direct bandgap band structure
    kpts = np.linspace(0, 1, 100)

    # Valence band maximum at Γ (k=0.5 in our path)
    Eg = 3.4  # GaN bandgap (eV)

    bands = []
    for i, offset in enumerate([-2.0, -1.5, -0.5, 0.0]):   # 4 valence bands
        vb = offset - 1.5 * np.sin(np.pi * kpts)**2
        bands.append(vb)

    for i, offset in enumerate([0.0, 0.5, 1.5]):             # 3 conduction bands
        cb = Eg + offset + 2.0 * np.sin(np.pi * kpts)**2
        bands.append(cb)

    return kpts, bands

kpts, bands = gan_schematic_bands()

fig, ax = plt.subplots(figsize=(6, 7))
for b in bands:
    ax.plot(kpts, b, 'b-', linewidth=1.5)

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', label='VBM')
ax.axhline(3.4, color='gray', linewidth=0.8, linestyle=':', label='CBM')
ax.annotate('', xy=(0.5, 3.4), xytext=(0.5, 0),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.52, 1.7, f'Eg = 3.4 eV
(direct)', color='red', fontsize=11)

ax.set_xticks([0, 0.33, 0.5, 0.66, 1.0])
ax.set_xticklabels(['Γ', 'M', 'K', 'Γ', 'A'], fontsize=12)
ax.set_ylabel('Energy (eV)', fontsize=12)
ax.set_title('Schematic GaN band structure
(direct bandgap at Γ)', fontsize=12)
ax.set_ylim(-4, 8)
ax.axvline(0.33, color='lightgray', lw=0.8)
ax.axvline(0.5, color='lightgray', lw=0.8)
ax.axvline(0.66, color='lightgray', lw=0.8)
plt.tight_layout()
plt.show()
print("In a DFT+GPAW calculation, replace this with calc_bs.band_structure().plot()")


## Density of States

The **density of states (DOS)** $g(E)$ counts the number of electronic states per unit energy interval. The **projected DOS (PDOS)** resolves contributions from each atomic species or orbital — very useful for understanding defect states.

For a quantum emitter like the NV centre, we look for:
- In-gap states inside the host bandgap
- Localised character (predominantly on the N and surrounding C atoms)


In [ ]:
# Schematic DOS to illustrate the concept
E = np.linspace(-5, 10, 1000)

def gaussian(E, E0, sigma, weight=1.0):
    return weight * np.exp(-(E - E0)**2 / (2*sigma**2)) / (sigma * np.sqrt(2*np.pi))

# Host DOS (valence band + conduction band)
dos_host = (np.exp(-(E+2)**2/2) * (E < 0) +
            gaussian(E, -1.5, 0.5) * (E < 0) +
            gaussian(E, 5.0, 1.0) * (E > 3.4))

# Defect in-gap state
dos_defect = gaussian(E, 1.5, 0.08, weight=0.3)

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(E, dos_host, alpha=0.4, color='steelblue', label='Host (C)')
ax.fill_between(E, dos_defect, alpha=0.8, color='red', label='Defect state (N+V)')
ax.axvline(0, color='gray', lw=1, ls='--')
ax.axvline(3.4, color='gray', lw=1, ls=':')
ax.annotate('Valence band
maximum', xy=(0, 0.5), xytext=(-3, 0.6),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('Conduction band
minimum', xy=(3.4, 0.1), xytext=(5, 0.4),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('Energy (eV)', fontsize=12)
ax.set_ylabel('DOS (arb. units)', fontsize=12)
ax.set_title('Schematic PDOS for NV centre in diamond', fontsize=12)
ax.legend(fontsize=11)
ax.set_xlim(-5, 10)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()


## Key Points

- Electronic band structure requires two DFT steps: SCF then NSCF along k-path
- `cell.bandpath()` generates high-symmetry paths for any Bravais lattice
- Direct bandgap materials are required for efficient light emission
- The PDOS reveals in-gap defect states relevant for quantum emitters
- Standard DFT (PBE) **underestimates** bandgaps; use HSE06 or G₀W₀ for accurate gaps

## Exercise 6.1

Using the schematic DOS plot above as a template, modify the code to represent a **boron vacancy in hBN** (bandgap ~6 eV, in-gap state at ~4 eV). Change the labels and colours accordingly.

## Exercise 6.2 (Research)

Look up the calculated bandgaps of GaN, AlN, and hBN computed with PBE and HSE06 in the literature or Materials Project. By how much does PBE underestimate the gap in each case? Why does this matter for simulating quantum optical transitions?


## Exercise 7.3 — Coursework Preparation: Electronic Structure of Your Defect

Using the GPAW code template from this tutorial, set up (but do not necessarily run) a 
spin-polarised DFT calculation for your coursework defect system.

Write a Python script `coursework_dos.py` that:

1. Reads your `coursework_defect_relaxed.xyz`
2. Sets the correct charge state and initial magnetic moments
3. Configures a GPAW calculator with appropriate settings (mode="lcao", basis="dzp", 
   xc="PBE", spinpol=True)
4. Sets up an output `.gpw` file for later analysis

Answer these questions in a markdown cell (use the schematic DOS plots and literature values 
if you cannot run the calculation):

- What spin state do you expect for your defect? How many unpaired electrons?
- Where (roughly) do you expect the defect level in the bandgap? Estimate from the ZPL 
  energy in the literature.
- What is the bandgap of your host material (PBE and experimental)? How large is the 
  PBE underestimate?
- Would you trust the defect level position from PBE for your system? Justify your answer.
